**Phase 2:  Silver Thread Processing (with resume + ADLS sync)**

- Synapse Notebook: 01_silver_processing
- Silver thread processing with ADLS sync every 10 threads and resume support.
- Trigger: After 01_ingestion completes (or manual)
- Input:   bronze/emails/, bronze/attachments/
- Output:  silver_llm/ on ADLS


**Enviroment variables setup**

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<YOUR_OPENAI_RESOURCE>.services.ai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-01-01-preview"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["ADLS_STORAGE_ACCOUNT"] = "<YOUR_ADLS_ACCOUNT>"
os.environ["ADLS_CONTAINER"] = "<YOUR_ADLS_CONTAINER>"
os.environ["ADLS_STORAGE_KEY"] = "<YOUR_ADLS_KEY>"
os.environ["COSMOS_GREMLIN_ENDPOINT"] = "wss://<YOUR_COSMOS_ACCOUNT>.gremlin.cosmos.azure.com:443/"
os.environ["COSMOS_GREMLIN_KEY"] = "<YOUR_COSMOS_KEY>"
os.environ["COSMOS_DATABASE"] = "email_cosmos_db"
os.environ["COSMOS_GRAPH"] = "email_cosmos_graph"
os.environ["COSMOS_NOSQL_ENDPOINT"] = ""
os.environ["COSMOS_NOSQL_KEY"] = ""
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://<YOUR_SEARCH_SERVICE>.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"] = "<YOUR_SEARCH_KEY>"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "kg-chunks"
os.environ["AZURE_SEARCH_ENTITY_INDEX_NAME"] = "kg-entities"
os.environ["AZURE_SEARCH_COMMUNITY_INDEX_NAME"] = "kg-communities"
os.environ["PIPELINE_MODE"] = "llm"
os.environ["PROCESS_ATTACHMENTS"] = "false"
print("Environment variables set.")

**Download code from ADLS**

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

service = DataLakeServiceClient(
    account_url=f"https://{os.environ['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net",
    credential=os.environ["ADLS_STORAGE_KEY"]
)

fs = service.get_file_system_client(os.environ["ADLS_CONTAINER"])

for prefix in ["code/src", "code/config"]:
    for p in fs.get_paths(path=prefix, recursive=True):
        if not p.is_directory:
            local_path = p.name.replace("code/", "", 1)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            with open(local_path, "wb") as f:
                file_client = fs.get_file_client(p.name)
                f.write(file_client.download_file().readall())

sys.path.insert(0, ".")
src_count = len(os.listdir("src")) if os.path.exists("src") else 0
config_count = len(os.listdir("config")) if os.path.exists("config") else 0
print(f"Downloaded: {src_count} items in src/, {config_count} items in config/")


**ADLS Adapter Init**

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sys.path.insert(0, ".")

from src.storage import ADLSAdapter

adapter = ADLSAdapter()

**Configrations for Bronze to Silver processing**

In [ ]:
MODE = os.getenv("PIPELINE_MODE", "llm")  # llm | local | hybrid
BRONZE_PATH = "bronze"
SILVER_PATH = f"silver_{MODE}"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 50
KG_STRATEGY = "llm" if MODE == "llm" else "spacy"  # spacy | llm | hybrid
REL_STRATEGY = "llm" if MODE == "llm" else "cooccurrence"  # cooccurrence | llm | hybrid
PROCESS_ATTACHMENTS = os.getenv("PROCESS_ATTACHMENTS", "true").lower() in ("true", "1", "yes")

print(f"Mode: {MODE}")
print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")
print(f"Attachments: {PROCESS_ATTACHMENTS}")

**Download Bronze + existing Silver checkpoint for resume**

In [ ]:
import shutil

LOCAL_BRONZE = "/tmp/pipeline/bronze"
LOCAL_SILVER = "/tmp/pipeline/silver"

# Clean Bronze (always re-download), but keep Silver if it exists (resume)
if os.path.exists(LOCAL_BRONZE):
    shutil.rmtree(LOCAL_BRONZE)
os.makedirs(LOCAL_BRONZE, exist_ok=True)
os.makedirs(LOCAL_SILVER, exist_ok=True)

# Download Bronze emails
email_files = adapter.list_files(f"{BRONZE_PATH}/emails", "*.json")
os.makedirs(f"{LOCAL_BRONZE}/emails", exist_ok=True)

for f in email_files:
    data = adapter.read_bytes(f)
    local_file = os.path.join(LOCAL_BRONZE, "emails", os.path.basename(f))
    with open(local_file, "wb") as fh:
        fh.write(data)

print(f"Downloaded {len(email_files)} Bronze emails to local temp")

# Download Bronze attachments if processing
if PROCESS_ATTACHMENTS:
    att_files = adapter.list_files(f"{BRONZE_PATH}/attachments", "*")
    os.makedirs(f"{LOCAL_BRONZE}/attachments", exist_ok=True)
    for f in att_files:
        data = adapter.read_bytes(f)
        local_file = os.path.join(LOCAL_BRONZE, "attachments", os.path.basename(f))
        rel_path = f.replace(f"{BRONZE_PATH}/", "", 1)
        local_file = os.path.join(LOCAL_BRONZE, rel_path)
        os.makedirs(os.path.dirname(local_file), exist_ok=True)
        with open(local_file, "wb") as fh:
            fh.write(data)
    print(f"Downloaded {len(att_files)} Bronze attachments")

# Download identity registry if exists
if adapter.exists("config/identity_registry.json"):
    reg_data = adapter.read_bytes("config/identity_registry.json")
    with open(f"{LOCAL_BRONZE}/identity_registry.json", "wb") as fh:
        fh.write(reg_data)
    print("Downloaded identity registry")

# Download existing Silver from ADLS for resume
silver_files = adapter.list_files(SILVER_PATH, "*.json")
if silver_files:
    for f in silver_files:
        rel_path = f.replace(f"{SILVER_PATH}/", "", 1)
        local_file = os.path.join(LOCAL_SILVER, rel_path)
        os.makedirs(os.path.dirname(local_file), exist_ok=True)
        with open(local_file, "wb") as fh:
            fh.write(adapter.read_bytes(f))
    print(f"Downloaded {len(silver_files)} existing Silver files for resume")
else:
    print("No existing Silver files on ADLS — starting fresh")

In [ ]:
import inspect
from src.silver.thread_aware_processor import ThreadAwareProcessor
source = inspect.getsource(ThreadAwareProcessor._process_attachments_separately)
print("circuit_breaker" in source)
print("skip_llm_for_attachment" in source)

**Test code**

**Build identity registry**

In [ ]:
from src.silver.identity_registry import IdentityRegistry

registry_path = f"{LOCAL_BRONZE}/identity_registry.json"
registry = IdentityRegistry()

# Load existing registry if downloaded from ADLS
if os.path.exists(registry_path):
    registry.load(registry_path)
    print(f"Loaded existing registry: {registry.identity_count} identities")

# Build/update from Bronze emails
stats = registry.build_from_bronze(LOCAL_BRONZE)
registry.save(registry_path)
print(f"Identity registry: {registry.identity_count} identities")


**test**

**Test Delete local silver**

**Run Thread-Aware Processing with ADLS sync callback**

**Multi-Workers with only upload newly processed threads. **

In [ ]:
from src.silver.thread_aware_processor import ThreadAwareProcessor
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import threading
import json

# Create processor
processor = ThreadAwareProcessor(
    bronze_path=LOCAL_BRONZE,
    silver_path=LOCAL_SILVER,
    processing_mode=MODE,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    kg_extractor_strategy=KG_STRATEGY,
    relationship_extractor_strategy=REL_STRATEGY,
    identity_registry=registry,
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    use_azure=True,
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
)

# Build threads
threads = processor.thread_grouper.group_from_bronze(str(processor.bronze_path))
print(f"Total threads: {len(threads)}")

# Load checkpoint
processed_ids = processor._load_checkpoint()
print(f"Already processed: {len(processed_ids)}")

threads_to_do = [
    (i, t) for i, t in enumerate(threads)
    if t.conversation_id not in processed_ids
]
print(f"Remaining: {len(threads_to_do)}")

# Pre-populate synced files tracker with existing files
silver_local = Path(LOCAL_SILVER)
_synced_files = set()
for existing_file in silver_local.rglob("*.json"):
    _synced_files.add(str(existing_file.relative_to(silver_local)))
print(f"Pre-existing files (skip re-upload): {len(_synced_files)}")

_sync_counter = [0]

def sync_new_files_to_adls():
    """Upload only new files to ADLS. Returns count of files uploaded."""
    count = 0
    for local_file in silver_local.rglob("*.json"):
        file_key = str(local_file.relative_to(silver_local))
        if file_key not in _synced_files:
            adls_path = f"{SILVER_PATH}/{file_key}"
            with open(local_file, "rb") as fh:
                adapter.write_bytes(adls_path, fh.read())
            _synced_files.add(file_key)
            count += 1
    return count

# Parallel processing
lock = threading.Lock()
pending_threads = []  # completed but not yet confirmed on ADLS
success_count = 0
fail_count = 0
SYNC_EVERY = 1  # sync after every 3 threads completed


def process_one(item):
    i, thread = item
    try:
        if thread.is_thread:
            processor._process_thread(thread)
        else:
            processor._process_single_email(thread)

        with lock:
            processor.stats["threads_processed"] += 1
            pending_threads.append(thread.conversation_id)
            n = processor.stats["threads_processed"]
            print(f"[{n}/{len(threads_to_do)}] Done: '{thread.subject[:50]}' ({thread.email_count} emails)")

            if len(pending_threads) >= SYNC_EVERY:
                # Step 1: Upload new files to ADLS FIRST
                _sync_counter[0] += 1
                file_count = sync_new_files_to_adls()

                # Step 2: THEN update checkpoint (files are confirmed on ADLS)
                for tid in pending_threads:
                    processed_ids.add(tid)
                processor._save_checkpoint(processed_ids)

                # Step 3: Upload checkpoint to ADLS
                checkpoint_file = silver_local / "checkpoint.json"
                if checkpoint_file.exists():
                    adapter.write_bytes(f"{SILVER_PATH}/checkpoint.json", checkpoint_file.read_bytes())

                print(f"  Sync #{_sync_counter[0]}: {file_count} files → ADLS, checkpoint: {len(processed_ids)} threads")
                pending_threads.clear()

        return True
    except Exception as e:
        with lock:
            processor.stats["errors"] = processor.stats.get("errors", 0) + 1
            print(f"[ERROR] '{thread.subject[:50]}': {e}")
        return False

WORKERS = 1
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(process_one, item): item for item in threads_to_do}
    for future in as_completed(futures):
        if future.result():
            success_count += 1
        else:
            fail_count += 1

# Final sync for any remaining pending threads
if pending_threads:
    _sync_counter[0] += 1
    file_count = sync_new_files_to_adls()
    for tid in pending_threads:
        processed_ids.add(tid)
    processor._save_checkpoint(processed_ids)
    checkpoint_file = silver_local / "checkpoint.json"
    if checkpoint_file.exists():
        adapter.write_bytes(f"{SILVER_PATH}/checkpoint.json", checkpoint_file.read_bytes())
    print(f"  Final sync #{_sync_counter[0]}: {file_count} files → ADLS, checkpoint: {len(processed_ids)} threads")
    pending_threads.clear()

print(f"\nComplete — {success_count} success, {fail_count} failed, {WORKERS} workers")
print(f"Stats: {json.dumps(processor.stats, indent=2, default=str)}")
